# Comparativo de Modelos Clássicos (Sem Data Leakage)\n
Neste notebook, vamos treinar os modelos clássicos usados no Kaggle (XGBoost, Random Forest e Ridge) usando a **metodologia correta** (Treino em Motores 1-10, Teste em Motores 11-20) e com as mesmas 54 features que usamos na MLP. O objetivo é provar que a MLP profunda é de fato o melhor regressor para esse problema quando não há trapaça nos dados.

In [1]:
!pip install -q xgboost

In [2]:
import os
import h5py
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Scikit-learn e Modelos Clássicos
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor

## 1. Carregando os Dados e Gerando Features Temporais (54 Colunas)

In [3]:
# Carregar arquivo
filename = 'data/N-CMAPSS_DS02-006.h5'
with h5py.File(filename, 'r') as hdf:
    W_dev = np.array(hdf.get('W_dev'))
    X_s_dev = np.array(hdf.get('X_s_dev'))
    Y_dev = np.array(hdf.get('Y_dev'))
    A_dev = np.array(hdf.get('A_dev'))
    
    W_test = np.array(hdf.get('W_test'))
    X_s_test = np.array(hdf.get('X_s_test'))
    Y_test = np.array(hdf.get('Y_test'))
    A_test = np.array(hdf.get('A_test'))

def create_temporal_features_safe(W, X_s, Y, A, window=20):
    matriz_base = np.concatenate((W, X_s), axis=1).astype('float32')
    df = pd.DataFrame(matriz_base)
    df['unit'] = A[:, 0].astype('float32')
    df['RUL'] = Y.flatten().astype('float32')
    
    print("Calculando médias e variâncias...")
    df_mean = df.groupby('unit').rolling(window=window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index().astype('float32')
    df_var = df.groupby('unit').rolling(window=window, min_periods=1).var().fillna(0).reset_index(level=0, drop=True).sort_index().astype('float32')
    
    df_mean = df_mean[df.columns[:-2]]
    df_var = df_var[df.columns[:-2]]
    df_raw = pd.DataFrame(matriz_base)
    
    X_temporal = pd.concat([df_raw, df_mean, df_var], axis=1).values.astype('float32')
    y_labels = df['RUL'].values.astype('float32')
    
    del df, df_mean, df_var, df_raw, matriz_base
    gc.collect()
    
    return X_temporal, y_labels

print("Processando Treino Oficial...")
X_train_raw, y_train_full = create_temporal_features_safe(W_dev, X_s_dev, Y_dev, A_dev, window=20)

print("Processando Teste Oficial...")
X_test_raw, y_test = create_temporal_features_safe(W_test, X_s_test, Y_test, A_test, window=20)

del W_dev, X_s_dev, Y_dev, A_dev
del W_test, X_s_test, Y_test, A_test
gc.collect()

Processando Treino Oficial...
Calculando médias e variâncias...
Processando Teste Oficial...
Calculando médias e variâncias...


0

## 2. Escalonamento e Amostragem (Igual à MLP)

In [4]:
# Escalonamento
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw).astype('float32')
X_test_scaled = scaler.transform(X_test_raw).astype('float32')

del X_train_raw, X_test_raw
gc.collect()

print("Amostrando 40% do treino para viabilizar os modelos clássicos...")
X_train_shuf, _, y_train_shuf, _ = train_test_split(
    X_train_scaled, y_train_full, train_size=0.40, random_state=42
)

del y_train_full, X_train_scaled
gc.collect()

Amostrando 40% do treino para viabilizar os modelos clássicos...


0

## 3. Treinamento e Avaliação dos Modelos Clássicos

In [ ]:
import joblib

modelos = {
    "Ridge Regression": Ridge(alpha=1.0),
    "Random Forest (100 árvores)": RandomForestRegressor(n_estimators=100, max_depth=15, n_jobs=-1, random_state=42),
    "XGBoost Regressor": XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.1, n_jobs=-1, random_state=42)
}

resultados = []

for nome, modelo in modelos.items():
    print(f"\n{'='*40}\nTreinando {nome}...")
    modelo.fit(X_train_shuf, y_train_shuf)
    
    print(f"Avaliando {nome} no conjunto de TESTE (Motores 11-20)...")
    y_pred = modelo.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    print(f"MAE: {mae:.2f} | R²: {r2:.4f}")
    resultados.append({'Modelo': nome, 'MAE': mae, 'R2': r2})
    
    # --- NOVO: SALVANDO O MODELO ---
    # Limpando o nome para não dar erro no arquivo (Tirando espaços e parênteses)
    nome_arquivo = nome.replace(" ", "_").replace("(", "").replace(")", "").lower()
    caminho_modelo = f"data/modelo_{nome_arquivo}.pkl"
    joblib.dump(modelo, caminho_modelo)
    print(f"Modelo salvo em: {caminho_modelo}")
    
    # Limpar memória
    del modelo
    gc.collect()

# --- NOVO: SALVANDO O NORMALIZADOR ---
joblib.dump(scaler, 'data/scaler_baselines.pkl')
print("Normalizador salvo em: data/scaler_baselines.pkl")

print("\n" + "="*50)
print("🏆 RESUMO COMPARATIVO 🏆")
df_res = pd.DataFrame(resultados).sort_values(by='R2', ascending=False).reset_index(drop=True)
display(df_res)
print("\n(Nota: A nossa Deep MLP de 256 neurônios obteve R² de ~0.85 nesse exato mesmo dataset!)")



 Treinando Ridge Regression...


/home/anderson/miniconda3/envs/intelligent_systems/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 9.234211728603725e-10.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Avaliando Ridge Regression no conjunto de TESTE (Motores 11-20)...
MAE: 9.97 | R²: 0.6320

 Treinando Random Forest (100 árvores)...
Avaliando Random Forest (100 árvores) no conjunto de TESTE (Motores 11-20)...
MAE: 7.55 | R²: 0.7318

 Treinando XGBoost Regressor...
Avaliando XGBoost Regressor no conjunto de TESTE (Motores 11-20)...
MAE: 6.96 | R²: 0.7834

🏆 RESUMO COMPARATIVO 🏆


,Modelo,MAE,R2
0,XGBoost Regressor,6.960729,0.783424
1,Random Forest (100 árvores),7.548914,0.731849
2,Ridge Regression,9.970874,0.632026



 (Nota: A nossa Deep MLP de 256 neurônios obteve R² de ~0.85 nesse exato mesmo dataset!)
